In [ ]:
# composite image generation by copy-and-paste followed by image blending.
from libcom import get_composite_image
from libcom.utils.process_image import make_image_grid, draw_bbox_on_image
import cv2
test_dir = '../tests/generate_composite_source/'
img_list = ['1.jpg']
bbox_list = [[1000, 895, 1480, 1355]]
for i,img_name in enumerate(img_list):
    bg_img  = test_dir + 'background/' + img_name
    bbox    = bbox_list[i] # x1,y1,x2,y2
    fg_img  = test_dir + 'foreground/' + img_name
    fg_mask = test_dir + 'foreground_mask/' + img_name.replace('.jpg', '.png')
    # generate composite images by naive methods
    comp_img1, comp_mask1 = get_composite_image(fg_img, fg_mask, bg_img, bbox, 'none')
    comp_img2, comp_mask2 = get_composite_image(fg_img, fg_mask, bg_img, bbox, 'gaussian')
    comp_img3, comp_mask3 = get_composite_image(fg_img, fg_mask, bg_img, bbox, 'poisson')
    vis_list = [bg_img, fg_img, comp_img1, comp_mask1, comp_img2, comp_mask2, comp_img3, comp_mask3]
    # visualization results
    grid_img  = make_image_grid(vis_list, cols=4)
    cv2.imwrite(f'../docs/_static/image/generatecomposite_result{i+1}.jpg', grid_img)

In [ ]:
# color-transfer: transfer foreground color to fit background scene.
from libcom import color_transfer
from libcom.utils.process_image import make_image_grid
import cv2
comp_img1  = '../tests/color_transfer_source/composite/1.jpg'
comp_mask1 = '../tests/color_transfer_source/composite_mask/1.png'
trans_img1 = color_transfer(comp_img1, comp_mask1)
# visualization results
grid_img  = make_image_grid([comp_img1, comp_mask1, trans_img1], cols=3)
cv2.imwrite('../docs/_static/image/colortransfer_result1.jpg', grid_img)

In [ ]:
# predict harmony score for a composite image.
from libcom import HarmonyScoreModel
from libcom.utils.process_image import make_image_grid
import cv2
net = HarmonyScoreModel(device=0, model_type='BargainNet')
test_dir   = '../tests/harmony_score_prediction_source/'
img_names  = ['vaulted-cellar-247391_inharm.jpg', 'ameland-5651866_harm.jpg']
vis_list,scores = [], []
for img_name in img_names:
    comp_img  = test_dir + 'composite/' + img_name
    comp_mask = test_dir + 'composite_mask/' + img_name
    score     = net(comp_img, comp_mask)
    vis_list += [comp_img, comp_mask]
    scores.append(score)
grid_img  = make_image_grid(vis_list, text_list=[f'harmony_score:{scores[0]:.2f}', 'composite-mask', f'harmony_score:{scores[1]:.2f}', 'composite-mask'])
cv2.imwrite('../docs/_static/image/harmonyscore_result1.jpg', grid_img)

In [ ]:
# Object Placement Assessment (OPA) score prediction
from libcom import OPAScoreModel
from libcom import get_composite_image
from libcom.utils.process_image import make_image_grid
import cv2
net = OPAScoreModel(device=0, model_type='SimOPA')
test_dir  = '../tests/opa_score_prediction_source/'
bg_img    = test_dir + 'background/1.jpg'
fg_img    = test_dir + 'foreground/1.jpg'
fg_mask   = test_dir + 'foreground_mask/1.png'
bbox_list = [[1000, 895, 1480, 1355], [1000, 300, 1480, 760]]
comp1, comp_mask1 = get_composite_image(fg_img, fg_mask, bg_img, bbox_list[0])
comp2, comp_mask2 = get_composite_image(fg_img, fg_mask, bg_img, bbox_list[1])
score1 = net(comp1, comp_mask1)
score2 = net(comp2, comp_mask2)
grid_img  = make_image_grid([comp1, comp_mask1, comp2, comp_mask2], text_list=[f'opa_score:{score1:.2f}', 'composite-mask', f'opa_score:{score2:.2f}', 'composite-mask'])
cv2.imwrite('../docs/_static/image/opascore_result1.jpg', grid_img)

In [ ]:
# Foreground Object Search (FOS) score prediction
from libcom.utils.process_image import make_image_grid
from libcom import FOSScoreModel
import cv2
import torch

task_name = 'fos_score_prediction'
MODEL_TYPE = 'FOS_D' # choose from 'FOS_D', 'FOS_E'
background = '../tests/fos_score_prediction_source/background/1.jpg'
fg_bbox    = [1000, 895, 1480, 1355] # x1,y1,x2,y2
foreground = '../tests/fos_score_prediction_source/foreground/1.jpg'
foreground_mask = '../tests/fos_score_prediction_source/foreground_mask/1.png'
composite_image = '../tests/fos_score_prediction_source/composite/1.jpg'
net = FOSScoreModel(device=0, model_type=MODEL_TYPE)
score = net(background, foreground, fg_bbox, foreground_mask=foreground_mask)
grid_img  = make_image_grid([background, foreground, composite_image], text_list=[f'fos_score:{score:.2f}'])
cv2.imshow('fos_score_demo', grid_img)
